# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² Extracellular Vesicle dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Croissant schema URL:** [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- **Dataset identifier:** `10.71728/senscience.vq4a-28xa`
- **Dataset title:** Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Title: {metadata.name}\n\nDescription: {metadata.description}\n\nVersion: {metadata.version}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

We enumerate the dataset's record sets, and for each, the available fields. All entities are referenced by their `@id` field.

In [ ]:
# List record sets and their fields by `@id`
print("Available record set @id's and their field @id's:")
record_set_ids = []
for rs in metadata.record_sets:
    print(f"- Record set: {rs['@id']} (name: {rs['name']})")
    record_set_ids.append(rs['@id'])
    fields = rs.get('fields', [])
    if not fields:
        print("    [No fields found]")
    else:
        for field in fields:
            print(f"    - Field: {field['@id']} (name: {field['name']}, type: {field.get('dataType', 'unknown')})")


## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

*Below, we demonstrate loading all records from the main record set (the largest/canonical record set—please adjust the code if your exploration in section 2 reveals a different main record set in your version of the dataset).*

In [ ]:
# For this dataset, let's use the first record set as the main set
if not record_set_ids:
    raise ValueError("No record sets found in the metadata!")

# Use the first available record set
main_record_set_id = record_set_ids[0]
print(f"Using main record set: {main_record_set_id}")

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

print('Column names in main record set:')
print(dataframes[main_record_set_id].columns.tolist())

print('\nFirst 5 rows:')
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Operations may include removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

Below, we select a numeric field (e.g., protein abundance, coverage, or molecular weight/@id) for analysis, filter for large values, normalize, and optionally group by a categorical variable (e.g., sample or protein class).

> _Replace the variable values below by selecting appropriate `@id` fields from your exploration in Section 2. This example demonstrates by using actual field names revealed above._

In [ ]:
# Choose a numeric field `@id` (update as needed)
df = dataframes[main_record_set_id]

# Fall back to commonly expected field names if nothing is found
possible_numeric_fields = [
    f for f in df.columns
    if (str(df[f].dtype) in ['int64', 'float64'] and 'abundance' in f.lower())
]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]  # Use the first numeric abundance-related field
else:
    # Arbitrarily pick the first numeric column
    numeric_fields = [f for f in df.columns if str(df[f].dtype) in ['int64', 'float64']]
    if not numeric_fields:
        raise ValueError('No numeric fields found in the main DataFrame!')
    numeric_field_id = numeric_fields[0]

print(f"Analyzing field: {numeric_field_id}")

# Choose a threshold relative to distribution
threshold = df[numeric_field_id].quantile(0.9)
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (90th percentile): {len(filtered_df)} records")
display(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / (filtered_df[numeric_field_id].std())
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Pick a group field for demonstration (e.g., by 'description' or similar categorical field)
possible_group_fields = [f for f in df.columns if any(x in f.lower() for x in ['description', 'sample', 'class', 'group'])]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped data (mean {numeric_field_id}) by {group_field}:")
    display(grouped_df.head())
else:
    print('No suitable group field found for grouping by categorical variable.')

## 5. Visualization

Visualize the distribution of the selected numeric field and compare group means if applicable.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field in the main record set
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

if group_field is not None:
    # Plot top 10 categories by mean value
    grouped_df_sorted = grouped_df.sort_values(by=numeric_field_id, ascending=False).head(10)
    plt.figure(figsize=(10, 4))
    sns.barplot(x=numeric_field_id, y=group_field, data=grouped_df_sorted, palette="viridis")
    plt.title(f"Top 10 {group_field} by mean {numeric_field_id}")
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR² Croissant dataset of extracellular vesicle proteins, explored its record sets and fields, and performed basic EDA.

- The dataset contains detailed protein measurements across several fields (the record set and field `@id`s were used for all referencing).
- Example filtering and normalization of protein abundances were demonstrated, and categorical breakdowns visualized if available.
- This workflow can be extended for more advanced biological or statistical analysis.

Feel free to further explore or process the data as suits your research aims! For more details on `mlcroissant`, see the [documentation](https://github.com/mlcommons/croissant).